<a href="https://colab.research.google.com/github/daiisntos/titanic-decision-tree-simulator/blob/main/_Titanic_%E2%80%93_Machine_Learning_from_Disaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🚢 Titanic – Machine Learning from Disaster

### 1. Introdução
O naufrágio do Titanic (1912) é um dos acidentes marítimos mais marcantes da história. Este projeto utiliza a base de dados do Kaggle para desenvolver um modelo de Machine Learning capaz de prever a sobrevivência dos passageiros com base em características como idade, sexo e classe.

### 2. Objetivo do Projeto
Investigar quais fatores influenciaram a sobrevivência e construir um modelo preditivo de classificação binária (`0`: não sobreviveu | `1`: sobreviveu).

### 3. Metodologia
1. **Análise Exploratória:** Investigação de padrões e dados ausentes.
2. **Preparação dos Dados:** Tratamento de valores nulos e variáveis categóricas.
3. **Modelagem:** Treinamento de algoritmos de Machine Learning.
4. **Avaliação:** Medição do desempenho com foco na acurácia (*Accuracy*).
5. **Simulação Interativa:** Validação prática de cenários em tempo real.

### 4. Abordagem
A análise baseia-se estritamente nas evidências encontradas nos dados, evitando suposições prévias e garantindo que o modelo reflita os padrões reais do histórico do navio.

In [45]:
import pandas as pd
import numpy as nm
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
list_to_passagers = pd.read_csv("test.csv")
print(list_to_passagers.head())



   PassengerId  Pclass                                          Name     Sex  \
0          892       3                              Kelly, Mr. James    male   
1          893       3              Wilkes, Mrs. James (Ellen Needs)  female   
2          894       2                     Myles, Mr. Thomas Francis    male   
3          895       3                              Wirz, Mr. Albert    male   
4          896       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female   

    Age  SibSp  Parch   Ticket     Fare Cabin Embarked  
0  34.5      0      0   330911   7.8292   NaN        Q  
1  47.0      1      0   363272   7.0000   NaN        S  
2  62.0      0      0   240276   9.6875   NaN        Q  
3  27.0      0      0   315154   8.6625   NaN        S  
4  22.0      1      1  3101298  12.2875   NaN        S  


In [47]:
print(list_to_passagers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB
None


In [48]:
print(list_to_passagers.shape)

(418, 11)


In [49]:
print(list_to_passagers.describe())

       PassengerId      Pclass         Age       SibSp       Parch        Fare
count   418.000000  418.000000  332.000000  418.000000  418.000000  417.000000
mean   1100.500000    2.265550   30.272590    0.447368    0.392344   35.627188
std     120.810458    0.841838   14.181209    0.896760    0.981429   55.907576
min     892.000000    1.000000    0.170000    0.000000    0.000000    0.000000
25%     996.250000    1.000000   21.000000    0.000000    0.000000    7.895800
50%    1100.500000    3.000000   27.000000    0.000000    0.000000   14.454200
75%    1204.750000    3.000000   39.000000    1.000000    0.000000   31.500000
max    1309.000000    3.000000   76.000000    8.000000    9.000000  512.329200


In [50]:
print(list_to_passagers.isna().sum())

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


Pela análise inicial, iremos desconsiderar a coluna **Cabin** e concentrar nossa análise nas demais variáveis, como **Fare**, utilizando como referência a mediana das tarifas pagas por passageiros da mesma classe.

Em seguida, vamos analisar com mais atenção a coluna **Age (Idade)**, que apresenta 86 valores ausentes. Para tratar esses dados faltantes, decidi utilizar a média das idades de acordo com o título presente no nome de cada passageiro (Mr., Mrs., Miss., Master etc.).

Essa escolha pode influenciar os resultados nas próximas etapas da análise e do modelo preditivo, mas, por enquanto, seguiremos com essa abordagem, que me pareceu a mais adequada para estimar as idades ausentes.

In [51]:
# 1. Extrair o título da coluna Name usando REGEX, o regex procura qualquer sequência de letras entre um espaço e um ponto
list_to_passagers['Title'] = list_to_passagers['Name'].str.extract(
    r'([A-Za-z]+)\.', expand=False
)

# Vamos ver quais títulos aparecem e quantas vezes cada um aparece?
print(list_to_passagers['Title'].value_counts())

# 2. Preencher os valores nulos da coluna Age usando a mediana da idade do respectivo Título
list_to_passagers['Age'] = list_to_passagers['Age'].fillna(
    list_to_passagers.groupby('Title')['Age'].transform('median')
)

# 3. Conferir se ainda sobrou algum valor nulo em Age
print('Valores nulos em Age após o preenchimento:', list_to_passagers['Age'].isna().sum())

Title
Mr        240
Miss       78
Mrs        72
Master     21
Col         2
Rev         2
Ms          1
Dr          1
Dona        1
Name: count, dtype: int64
Valores nulos em Age após o preenchimento: 1


In [52]:
list_to_passagers[list_to_passagers['Age'].isna()]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title
88,980,3,"O'Donoghue, Ms. Bridget",female,NaN,0,0,364856,7.75,NaN,Q,Ms


Como observamos que Bridget é uma mulher, o ideal neste caso, foi padronizar seu título para seguir o mesmo padrão utilizado nos demais registros femininos. Em seguida, devemos preencher o valor ausente da coluna **Age** utilizando a mediana das idades das passageiras que possuem o mesmo título.

In [53]:
# Extrair o título da coluna Name usando REGEX
list_to_passagers['Title'] = list_to_passagers['Name'].str.extract(
    r'([A-Za-z]+)\.', expand=False
)

# Padronizar títulos equivalentes (como 'Ms' virando 'Miss')
list_to_passagers['Title'] = list_to_passagers['Title'].replace({'Ms': 'Miss'})

# Preencher os valores nulos da coluna Age usando a mediana do respectivo Título
list_to_passagers['Age'] = list_to_passagers['Age'].fillna(
    list_to_passagers.groupby('Title')['Age'].transform('median')
)

# Conferir se ainda sobrou algum valor nulo em Age
print('Valores nulos em Age após o preenchimento:', list_to_passagers['Age'].isna().sum())

Valores nulos em Age após o preenchimento: 0


In [54]:

# Preencher os valores nulos da coluna Fare usando a mediana da Pclass do respectivo grupo em que ele pertence
list_to_passagers['Fare'] = list_to_passagers['Fare'].fillna(
    list_to_passagers.groupby('Pclass')['Fare'].transform('median')
)



In [55]:
print(list_to_passagers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          418 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         418 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
 11  Title        418 non-null    object 
dtypes: float64(2), int64(4), object(6)
memory usage: 39.3+ KB
None


In [56]:
list_to_passagers['Age'] = list_to_passagers['Age'].astype(int)

In [57]:
print(list_to_passagers['Age'].astype(int))

0      34
1      47
2      62
3      27
4      22
       ..
413    28
414    39
415    38
416    28
417     7
Name: Age, Length: 418, dtype: int64


In [58]:
print(list_to_passagers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          418 non-null    int64  
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         418 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
 11  Title        418 non-null    object 
dtypes: float64(1), int64(5), object(6)
memory usage: 39.3+ KB
None


In [61]:
# Neste bloco decidi fazer com que o Sexo retornasse números binários
list_to_passagers['Sex'] = list_to_passagers['Sex'].map({'female': 0, 'male': 1})


In [77]:
# @title 🚢 Simulador Interativo de Sobrevivência do Titanic
# Escolha a classe social do passageiro (1 = Primeira, 2 = Segunda, 3 = Terceira)
pclass = 3  # @param {type:"slider", min:1, max:3, step:1}

# Escolha o sexo (0 para feminino, 1 para masculino)
sex = 1  # @param ["0", "1"] {type:"raw"}

# Escolha a idade do passageiro
age = 35  # @param {type:"slider", min:1, max:80, step:1}

import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Carregamento e tratamento básico dos dados
df_treino = pd.read_csv('train.csv')
df_treino['Age'] = df_treino['Age'].fillna(df_treino['Age'].median())

if df_treino['Sex'].dtype == 'object':
  df_treino['Sex'] = df_treino['Sex'].map({'female': 0, 'male': 1})

# 1. Definimos APENAS as 3 features principais
features = ['Pclass', 'Sex', 'Age']
X = df_treino[features]
y = df_treino['Survived']

# 2. Treinamos o modelo
modelo_simulador = DecisionTreeClassifier(max_depth=12, random_state=42)
modelo_simulador.fit(X, y)

# 3. Preparamos o passageiro hipotético com exatamente os 3 valores correspondentes
passageiro_hipotetico = [[pclass, sex, age]]

# 4. O modelo faz a previsão
previsao = modelo_simulador.predict(passageiro_hipotetico)[0]

# 5. Mostramos o resultado
print('----------------------------------------')
print('📊 Dados do Passageiro Simulado:')
print(f' • Classe (Pclass): {pclass}')
print(f" • Sexo: {'Feminino' if sex == 0 else 'Masculino'}")
print(f' • Idade: {age} anos')
print('----------------------------------------')

if previsao == 1:
  print(
      '**RESULTADO:** Este passageiro teria **SOBREVIVIDO**'
  )
else:
  print(
      '**RESULTADO:** Infelizmente, este passageiro **NÃO TERIA'
      ' SOBREVIVIDO**'
  )
print('----------------------------------------')

----------------------------------------
📊 Dados do Passageiro Simulado:
 • Classe (Pclass): 3
 • Sexo: Masculino
 • Idade: 35 anos
----------------------------------------
**RESULTADO:** Infelizmente, este passageiro **NÃO TERIA SOBREVIVIDO**
----------------------------------------


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


Com o modelo pronto, testei a acúracia do modelo teste com base no arquivo da Kagle train

In [78]:
#  Avaliação de Acurácia do Modelo com o train.csv
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

# 1. Carregamos e tratamos os dados do train.csv
df_avaliacao = pd.read_csv('train.csv')
df_avaliacao['Age'] = df_avaliacao['Age'].fillna(df_avaliacao['Age'].median())

if df_avaliacao['Sex'].dtype == 'object':
  df_avaliacao['Sex'] = df_avaliacao['Sex'].map({'female': 0, 'male': 1})

# 2. Definimos as mesmas 3 features e o target
features = ['Pclass', 'Sex', 'Age']
X = df_avaliacao[features]
y = df_avaliacao['Survived']

# 3. Criamos o modelo de árvore de decisão (com a mesma profundidade que ajustámos)
modelo_avaliacao = DecisionTreeClassifier(max_depth=12, random_state=42)

# 4. Aplicamos a Validação Cruzada (cv=5 divide os dados em 5 partes para testar de forma robusta)
scores = cross_val_score(modelo_avaliacao, X, y, cv=5, scoring='accuracy')

# 5. Mostramos os resultados detalhados e a média geral
print('----------------------------------------')
print('📈 Relatório de Desempenho do Modelo:')
print('----------------------------------------')
for i, score in enumerate(scores):
  print(f' • Validação parte {i+1}: {score * 100:.2f}% de acerto')

print('----------------------------------------')
print(f'🎯 **Acurácia Média Final:** {scores.mean() * 100:.2f}%')
print('----------------------------------------')

----------------------------------------
📈 Relatório de Desempenho do Modelo:
----------------------------------------
 • Validação parte 1: 77.09% de acerto
 • Validação parte 2: 79.78% de acerto
 • Validação parte 3: 80.34% de acerto
 • Validação parte 4: 79.21% de acerto
 • Validação parte 5: 82.02% de acerto
----------------------------------------
🎯 **Acurácia Média Final:** 79.69%
----------------------------------------
